# MAT353 — Doğrusal Regresyon Deneyleri (Tek Notebook)

Tüm kod bu notebook içinde (modül hücreleri yukarıda). Adımlar:
1. Hücreleri sırayla çalıştırın; harici `src/` yok.
2. Sentetik deneyler: p taraması, n/d ölçekleme, GD, ε sweep.
3. Gerçek veri (California Housing) deneyini çalıştırın.
4. Şekilleri `../outputs/figures`, tabloları `../outputs/tables` altına kaydedin.


In [ ]:
# Modül kodları (data, solvers, gradcheck, metrics, utils)
from __future__ import annotations
from typing import Tuple, Callable, Dict, List, Iterable, Iterator
import contextlib
import time
import numpy as np
from numpy.linalg import inv, solve, cond
from scipy.linalg import lstsq

# data.py

def make_synthetic_linear(
    n: int,
    d: int,
    p: int,
    noise_std: float = 0.5,
    seed: int | None = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    x1 = rng.normal(size=(n, 1))
    features = [x1]
    if d >= 2:
        x2 = x1 + (10.0 ** (-p)) * rng.normal(size=(n, 1))
        features.append(x2)
    if d > 2:
        rest = rng.normal(size=(n, d - 2))
        features.append(rest)
    X = np.hstack(features)
    theta_true = rng.normal(size=(d,))
    noise = noise_std * rng.normal(size=n)
    y = X @ theta_true + noise
    return X, y, theta_true


def train_val_split(
    X: np.ndarray,
    y: np.ndarray,
    val_ratio: float = 0.2,
    seed: int | None = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    idx = rng.permutation(n)
    split = int(n * (1 - val_ratio))
    train_idx, val_idx = idx[:split], idx[split:]
    return X[train_idx], X[val_idx], y[train_idx], y[val_idx]


def standardize(
    X_train: np.ndarray, X_val: np.ndarray
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0) + 1e-12
    X_train_std = (X_train - mean) / std
    X_val_std = (X_val - mean) / std
    return X_train_std, X_val_std, mean, std

# solvers.py

def normal_eq_inverse(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    xtx = X.T @ X
    xty = X.T @ y
    return inv(xtx) @ xty


def normal_eq_solve(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    xtx = X.T @ X
    xty = X.T @ y
    return solve(xtx, xty)


def least_squares_qr(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    theta, *_ = lstsq(X, y)
    return theta


def gradient_descent(
    X: np.ndarray,
    y: np.ndarray,
    alpha: float = 1e-3,
    max_iter: int = 10_000,
    tol: float = 1e-6,
    log_every: int = 10,
    callback: Callable[[int, float, float], None] | None = None,
) -> Tuple[np.ndarray, Dict[str, List[float]]]:
    n, d = X.shape
    theta = np.zeros(d)
    losses: List[float] = []
    grad_norms: List[float] = []
    for k in range(1, max_iter + 1):
        residual = X @ theta - y
        loss = 0.5 / n * np.dot(residual, residual)
        grad = (X.T @ residual) / n
        grad_norm = np.linalg.norm(grad)
        theta -= alpha * grad
        if k % log_every == 0 or k == 1:
            losses.append(loss)
            grad_norms.append(grad_norm)
            if callback:
                callback(k, loss, grad_norm)
        if grad_norm < tol:
            break
    history = {"losses": losses, "grad_norms": grad_norms}
    return theta, history

# gradcheck.py

def J_mse(theta: np.ndarray, X: np.ndarray, y: np.ndarray) -> float:
    n = X.shape[0]
    residual = X @ theta - y
    return 0.5 / n * np.dot(residual, residual)


def grad_mse_analytic(theta: np.ndarray, X: np.ndarray, y: np.ndarray) -> np.ndarray:
    n = X.shape[0]
    return (X.T @ (X @ theta - y)) / n


def grad_numeric_central(
    J: Callable[[np.ndarray], float],
    theta: np.ndarray,
    eps: float = 1e-4,
) -> np.ndarray:
    grad = np.zeros_like(theta)
    for i in range(theta.size):
        e = np.zeros_like(theta)
        e[i] = 1.0
        grad[i] = (J(theta + eps * e) - J(theta - eps * e)) / (2 * eps)
    return grad


def epsilon_sweep(
    theta: np.ndarray,
    X: np.ndarray,
    y: np.ndarray,
    eps_list: Iterable[float],
) -> Tuple[np.ndarray, np.ndarray]:
    analytic = grad_mse_analytic(theta, X, y)
    eps_array = np.array(list(eps_list))
    relerrs = []
    for eps in eps_array:
        numeric = grad_numeric_central(lambda t: J_mse(t, X, y), theta, eps)
        denom = np.maximum(np.abs(analytic), np.abs(numeric))
        relerr = np.linalg.norm((analytic - numeric) / np.maximum(denom, 1e-15))
        relerrs.append(relerr)
    return eps_array, np.array(relerrs)

# metrics.py

def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def cond_xtx(X: np.ndarray) -> float:
    return float(cond(X.T @ X))

# utils.py
@contextlib.contextmanager
def elapsed_timer() -> Iterator[callable]:
    start = time.perf_counter()
    def _elapsed():
        return time.perf_counter() - start
    yield _elapsed

def set_seed(seed: int | None = None) -> np.random.Generator:
    return np.random.default_rng(seed)


In [13]:
import os
from pathlib import Path

ROOT = Path("..").resolve()
FIG_DIR = ROOT / "outputs" / "figures"
TAB_DIR = ROOT / "outputs" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
rng_global = np.random.default_rng(SEED)

print("Project root:", ROOT)
print("Figures ->", FIG_DIR)
print("Tables  ->", TAB_DIR)
print("Seed:", SEED)

Project root: /Users/omerfarukaltinova/Git_Projects/numerikproje
Figures -> /Users/omerfarukaltinova/Git_Projects/numerikproje/outputs/figures
Tables  -> /Users/omerfarukaltinova/Git_Projects/numerikproje/outputs/tables
Seed: 42


In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing

# Fonksiyonlar yukarıdaki modül hücresinden geliyor.

sns.set_theme(style="whitegrid")

# Sürüm/log bilgisi
versions = {
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": plt.matplotlib.__version__,
    "seaborn": sns.__version__,
}
print("Versions:", versions)


Versions: {'numpy': '2.3.5', 'pandas': '2.3.3', 'matplotlib': '3.10.8', 'seaborn': '0.13.2'}


In [15]:
# Deney parametreleri ve log ayarları
p_values = range(0, 13)
n_values = [1000, 5000, 10000, 20000, 50000]
d_values = [10, 50, 100, 200, 500]
gd_params = {"alpha": 1e-3, "max_iter": 5000, "tol": 1e-6}
meta = {
    "seed": SEED,
    "p_values": list(p_values),
    "n_values": n_values,
    "d_values": d_values,
    "gd_params": gd_params,
}
with open(TAB_DIR / "experiment_meta.json", "w", encoding="utf-8") as f:
    import json
    json.dump(meta, f, ensure_ascii=False, indent=2)
print("Saved meta ->", TAB_DIR / "experiment_meta.json")

Saved meta -> /Users/omerfarukaltinova/Git_Projects/numerikproje/outputs/tables/experiment_meta.json


### Ölçekleme (n) — runtime ve RMSE
- n: 1k, 5k, 10k, 20k, 50k (d=20, p=4)
- solve vs GD (standardize)
- Tablo + runtime grafiği kaydı

In [16]:
# Ölçekleme (n) deneyi
rows_n = []
for n in [1000, 5000, 10000, 20000, 50000]:
    X, y, theta_true = make_synthetic_linear(n=n, d=20, p=4, noise_std=0.5, seed=SEED)
    X_train, X_val, y_train, y_val = train_val_split(X, y, val_ratio=0.2, seed=SEED)
    # solve
    with elapsed_timer() as t:
        theta_solve = normal_eq_solve(X_train, y_train)
    solve_time = t()
    # GD (standardize)
    X_train_std, X_val_std, mean, std = standardize(X_train, X_val)
    with elapsed_timer() as t:
        theta_gd, history_gd = gradient_descent(
            X_train_std, y_train,
            alpha=gd_params["alpha"], max_iter=gd_params["max_iter"], tol=gd_params["tol"], log_every=200,
        )
    gd_time = t()
    rows_n.append(dict(n=n, method="solve", rmse_train=rmse(y_train, X_train@theta_solve), rmse_val=rmse(y_val, X_val@theta_solve), runtime=solve_time))
    rows_n.append(dict(n=n, method="gd", rmse_train=rmse(y_train, X_train_std@theta_gd), rmse_val=rmse(y_val, X_val_std@theta_gd), runtime=gd_time))

n_df = pd.DataFrame(rows_n)
save_table(n_df, "scaling_n_table")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for m in ["solve", "gd"]:
    subset = n_df[n_df["method"] == m]
    ax[0].plot(subset["n"], subset["runtime"], marker="o", label=m)
    ax[1].plot(subset["n"], subset["rmse_val"], marker="o", label=m)
ax[0].set_xlabel("n")
ax[0].set_ylabel("runtime (s)")
ax[0].set_title("Runtime vs n")
ax[0].legend()
ax[1].set_xlabel("n")
ax[1].set_ylabel("RMSE val")
ax[1].set_title("Val RMSE vs n")
ax[1].legend()
save_fig(fig, "scaling_n_runtime_rmse")
plt.show()


TypeError: 'generator' object does not support the context manager protocol

In [ ]:
# Ölçekleme (d) deneyi
rows_d = []
for d in [10, 50, 100, 200, 500]:
    X, y, theta_true = make_synthetic_linear(n=2000, d=d, p=4, noise_std=0.5, seed=SEED)
    X_train, X_val, y_train, y_val = train_val_split(X, y, val_ratio=0.2, seed=SEED)
    # solve
    with elapsed_timer() as t:
        theta_solve = normal_eq_solve(X_train, y_train)
    solve_time = t()
    # GD (standardize)
    X_train_std, X_val_std, mean, std = standardize(X_train, X_val)
    with elapsed_timer() as t:
        theta_gd, history_gd = gradient_descent(
            X_train_std, y_train,
            alpha=gd_params["alpha"], max_iter=gd_params["max_iter"], tol=gd_params["tol"], log_every=200,
        )
    gd_time = t()
    rows_d.append(dict(d=d, method="solve", rmse_train=rmse(y_train, X_train@theta_solve), rmse_val=rmse(y_val, X_val@theta_solve), runtime=solve_time))
    rows_d.append(dict(d=d, method="gd", rmse_train=rmse(y_train, X_train_std@theta_gd), rmse_val=rmse(y_val, X_val_std@theta_gd), runtime=gd_time))

d_df = pd.DataFrame(rows_d)
save_table(d_df, "scaling_d_table")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for m in ["solve", "gd"]:
    subset = d_df[d_df["method"] == m]
    ax[0].plot(subset["d"], subset["runtime"], marker="o", label=m)
    ax[1].plot(subset["d"], subset["rmse_val"], marker="o", label=m)
ax[0].set_xlabel("d")
ax[0].set_ylabel("runtime (s)")
ax[0].set_title("Runtime vs d")
ax[0].legend()
ax[1].set_xlabel("d")
ax[1].set_ylabel("RMSE val")
ax[1].set_title("Val RMSE vs d")
ax[1].legend()
save_fig(fig, "scaling_d_runtime_rmse")
plt.show()


In [ ]:
# Yardımcılar: kayıt
import json

def save_table(df: pd.DataFrame, name: str):
    path = TAB_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    print("Saved table ->", path)


def save_fig(fig, name: str, tight: bool = True):
    if tight:
        fig.tight_layout()
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=200)
    print("Saved figure ->", path)

# Sürüm/log kaydı
versions_path = TAB_DIR / "versions.json"
with open(versions_path, "w", encoding="utf-8") as f:
    json.dump(versions, f, ensure_ascii=False, indent=2)
print("Saved versions ->", versions_path)


## Gradient Descent deneyi
- Standardize edilmiş özellikler ile GD koş.
- Kaydet: loss eğrisi, grad norm eğrisi, runtime, yakınsama durumu.

In [ ]:
# TODO: GD hiperparametre seçimi (alpha, max_iter, tol)
alpha = 1e-3
max_iter = 5000
tol = 1e-6

X, y, theta_true = make_synthetic_linear(n=2000, d=20, p=8, noise_std=0.5, seed=123)
X_train, X_val, y_train, y_val = train_val_split(X, y, val_ratio=0.2, seed=123)
X_train_std, X_val_std, mean, std = standardize(X_train, X_val)

with elapsed_timer() as t:
    theta_gd, history = gradient_descent(
        X_train_std, y_train,
        alpha=alpha,
        max_iter=max_iter,
        tol=tol,
        log_every=50,
    )
gd_time = t()

y_pred_train = X_train_std @ theta_gd
y_pred_val = X_val_std @ theta_gd

print("GD runtime (s):", gd_time)
print("RMSE train/val:", rmse(y_train, y_pred_train), rmse(y_val, y_pred_val))

# Loss ve grad norm eğrileri
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(history["losses"])
ax[0].set_title("Loss")
ax[0].set_xlabel("log step")
ax[0].set_ylabel("J(theta)")

ax[1].plot(history["grad_norms"])
ax[1].set_title("Grad norm")
ax[1].set_xlabel("log step")
ax[1].set_ylabel("||grad||")

fig.tight_layout()
plt.show()


## Gradient checking ve ε taraması
- Analitik vs sayısal gradyan (merkezi fark).
- ε listesi: 1e-1 .. 1e-10 (log ekseni önerilir).
- relerr grafiği beklenen U-şeklini vermeli.

In [ ]:
eps_list = [10.0 ** (-k) for k in range(1, 11)]

X, y, theta_true = make_synthetic_linear(n=2000, d=20, p=4, noise_std=0.5, seed=7)
theta0 = np.zeros(X.shape[1])

eps_array, relerr = epsilon_sweep(theta0, X, y, eps_list)

plt.figure(figsize=(5, 4))
plt.plot(eps_array, relerr, marker="o")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("epsilon")
plt.ylabel("relerr")
plt.title("Gradient checking: epsilon sweep")
plt.grid(True, which="both")
plt.show()


## Gerçek veri (opsiyonel)
- sklearn California Housing veya gömülü küçük CSV.
- Aynı metrik seti ile solve/GD kıyasla; koşullanma yorumlanabilir.